# How far apart two PFS values have to be

A PFS is computed from samples: the survey's respondents in every cell and, under
full answers, the model's sampled answers. This notebook measures how far the
difference between two models' scores moves when those samples are drawn again,
first on a synthetic example and then on two models of the paper.

**Resampling.** Every cell keeps its number of valid answers $n_i$ and its
observed shares $\hat p_i$, and a replicate draws its answers again,

$$X_i^{(b)} \sim \mathrm{Multinomial}(n_i, \hat p_i), \qquad \hat p_i^{(b)} = X_i^{(b)} / n_i,$$

which is drawing the cell's answers with replacement. The survey is redrawn once
per replicate and every model is scored against that same redrawn survey. A model
read from sampled answers is redrawn the same way, each model on its own; a model
read from next-token probabilities has no sample behind it and stays fixed.
Answers are redrawn, never the pairwise distances, which share cells. Every
replicate is scored by `group_fidelity`, so the scores are the paper's.

**The interval.** For a difference $\Delta$ between two models, the replicates
$\Delta^{(b)}$ give a spread. Redrawing a table that is already a sample adds a
second layer of sampling noise, and the scores are not linear in that noise:
accuracy and structure fall and the survey's dispersion grows. The replicates are
therefore centered beside the observed difference rather than on it, by a shift
$s = \bar\Delta^{*} - \hat\Delta$. The interval is the percentile interval of the
replicates, extended toward the observed difference by that shift,

$$\big[\; q_{0.025} - \max(s, 0),\;\; q_{0.975} + \max(-s, 0) \;\big],$$

so it holds both the central 95% of the replicates and the same spread placed
around the observed value. The tables keep the two parts as `percentile_low`,
`percentile_high` and `centered_low`, `centered_high`, the shift as `shift`, and
the share of replicates on each side of zero and tied at it. A difference is
`separated` when the interval excludes zero.

**Ties.** PFS is zero whenever structure is, so two models can tie at zero in a
replicate. When at least 2.5% of the replicates tie, a percentile of the
replicates is the tie itself, the interval ends at zero, and the difference is
not separated.

**The synthetic example.** One binary question, five groups, a survey and three
models, each share read from 2,000 answers. No two pairwise distances tie in any
of the four, so the structure of the example does not depend on how ties are
ranked.

**The two models.** Qwen3-VL-8B-Thinking and Gemma-4-31B-it as released, the
larger model of each family and the two the main text sets side by side, on the
four questions and both elicitation modes, first full-answer run. Their response
tables are read from the published dataset at a pinned revision, sixteen files,
and scored on the cells both models retain. Every question and mode has its own
seed, so a case redrawn alone gives the replicates it gives in the full run.

**A known population.** The last check draws datasets from a known population,
the synthetic shares and then the observed tables of the two models, resamples
each dataset as above, and counts how often its interval holds the difference
that the comparison reports on average.

Writes `population_fidelity_bootstrap_synthetic.csv`,
`population_fidelity_bootstrap_levels.csv`,
`population_fidelity_bootstrap_differences.csv`,
`population_fidelity_bootstrap_runs.csv`,
`population_fidelity_bootstrap_calibration.csv` and the plates under
`figures/fidelity/bootstrap/`.

In [1]:
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore", message="IProgress not found")

from culture.fidelity import GROUPS_TABLE
from culture.fidelity_bootstrap import (
    BOOTSTRAP_CALIBRATION_TABLE,
    BOOTSTRAP_DIFFERENCES_TABLE,
    BOOTSTRAP_INPUTS_TABLE,
    BOOTSTRAP_LEVELS_TABLE,
    BOOTSTRAP_RUNS_TABLE,
    RELEASED_REVISION,
    SURVEYED,
    bootstrap_plates,
    bootstrap_tables,
    calibration_summary,
    check_bootstrap,
    released_bootstrap,
    released_calibration,
    response_files,
    run_differences,
    synthetic_bootstrap,
    synthetic_calibration,
    synthetic_inputs,
)
from culture.tables import read_csv
from hub.cards import DATASET_REPO_ID
from hub.commands import DEFAULT_DOWNLOAD
from hub.fetch import download_files
from machine_bias_reproduction.config import OUTPUTS_ROOT
from machine_bias_reproduction.io_utils import sha256_file, write_csv

## The response tables of the two models

Only the sixteen consolidated tables are fetched, one by one, so a table the
revision lacks stops the notebook: next-token probabilities and full answers of
the two as-released models on the four questions. Where the same run is on disk
under `outputs/`, the fetched table is compared with it byte for byte.

In [2]:
files = response_files()
snapshot = download_files(DATASET_REPO_ID, DEFAULT_DOWNLOAD, files, revision=RELEASED_REVISION)
released_root = snapshot / "data"
print(len(files), "response tables at revision", RELEASED_REVISION[:7])

16 response tables at revision 3c908b5


In [3]:
local = [(snapshot / name, OUTPUTS_ROOT / Path(name).relative_to("data")) for name in files]
identical = sum(sha256_file(fetched) == sha256_file(kept) for fetched, kept in local if kept.is_file())
print(identical, "of", len(files), "tables equal the copy under outputs/")

16 of 16 tables equal the copy under outputs/


## The synthetic example

In [4]:
synthetic = synthetic_bootstrap()
print(synthetic.n_replicates, "replicates of", len(synthetic.observed), "models over", synthetic.n_cells, "groups")

10000 replicates of 3 models over 5 groups


## Two models on the World Values Survey

One process per question and mode, each on its own seed, so the replicates do not
depend on how many processes run nor on which other cases run beside them.

In [5]:
released = released_bootstrap(released_root)
print(len(released), "question and mode combinations,", next(iter(released.values())).n_replicates, "replicates each")

8 question and mode combinations, 10000 replicates each


## Reconciling against the groups table

Both models retain every cell on every question, so the observed scores are those
of `population_fidelity_groups.csv`. The check asserts it on every reading of
every case scored on the cells the groups table scores.

In [6]:
levels, differences = bootstrap_tables(synthetic, released)
groups = read_csv(GROUPS_TABLE)
assert groups is not None
matched = check_bootstrap(levels, groups)
print(matched, "of", int(levels["study"].eq(SURVEYED).sum()), "observed scores equal the groups table")

128 of 128 observed scores equal the groups table


## The tables and the plates

`population_fidelity_bootstrap_levels.csv` holds one row per study, question,
mode, model and reading; `population_fidelity_bootstrap_differences.csv` one row
per pair of models and reading. `study` is `synthetic` or `wvs`.
`population_fidelity_bootstrap_runs.csv` sets the two models side by side in
every full-answer run of the groups table, one row per run and reading.

On a plate each row is one comparison: the shaded shape is the distribution of
the replicate differences, the diamond the observed difference, the bar the
interval, and the diamond is solid when the interval excludes zero.

In [7]:
runs = run_differences(groups)
write_csv(synthetic_inputs(), BOOTSTRAP_INPUTS_TABLE)
write_csv(levels, BOOTSTRAP_LEVELS_TABLE)
write_csv(differences, BOOTSTRAP_DIFFERENCES_TABLE)
write_csv(runs, BOOTSTRAP_RUNS_TABLE)
plates = bootstrap_plates(synthetic, released)
print(len(levels), "level rows,", len(differences), "difference rows,", len(runs), "run rows,", len(plates), "plate files")

152 level rows, 80 difference rows, 48 run rows, 4 plate files


## A known population

The resampling is checked where the population is known. For the synthetic
example the population is the table of shares itself: 1,000 datasets are drawn
from it, each resampled 500 times. For the two models the observed tables stand
in for the population: 100 datasets per question and mode, each resampled 200
times. `expected` is the mean difference over the datasets, the value the
comparison reports on average; `population` is the difference of the tables the
datasets were drawn from. A dataset counts for an interval when the interval holds
`expected`. With 200 replicates an exact interval holds it in about 94% of the
datasets rather than 95%, and 100 datasets place a share within about two points.

In [8]:
drawn = pd.concat([synthetic_calibration(), released_calibration(released_root)], ignore_index=True)
calibrated = calibration_summary(drawn)
write_csv(calibrated, BOOTSTRAP_CALIBRATION_TABLE)
print(len(drawn), "dataset rows,", len(calibrated), "calibration rows")

11200 dataset rows, 40 calibration rows


## Reading the tables

**Synthetic example.** Model C minus Model A is +0.095 with an interval of
[+0.045, +0.136], separated from zero. Model B minus Model A is +0.024 with
[−0.025, +0.078], not separated, while the components behind it are: B is ahead
on adaptability by +0.229 [+0.098, +0.332] and behind on structure by −0.170
[−0.221, −0.061]. Over the 1,000 redrawn datasets C minus A is separated in 95.5%
and B minus A in 17.1%, and the interval holds the expected difference in 98.6%
and 98.2%.

**The two models.** Qwen3-VL-8B-Thinking minus Gemma-4-31B-it is separated from
zero on happiness (+0.341 under full answers, +0.273 under next-token
probabilities) and on social trust (+0.082, +0.111). On political ideology under
full answers the interval spans zero, +0.080 [−0.152, +0.339]. On political
ideology under next-token probabilities and on religious attendance under both
modes the interval ends at zero: one model scores zero, the other stands on a
structure of at most 0.015, and both score zero in 3.6% to 11.6% of the
replicates. The sign of the difference changes with the question, so the table is
a guide to reading a difference and not a ranking of the two models.

**The shift.** The replicates of the PFS difference sit up to 0.069 from the
observed one (political ideology, full answers). The second and third
full-answer runs fall inside the interval of the first run in 8 of 8 cases for
PFS, 7 for accuracy, 8 for adaptability and 8 for structure; inside the
percentile interval alone in 7, 0, 4 and 6.

**A known population.** Over the eight question and mode combinations the
interval holds the expected difference in 94% to 99% of the datasets for PFS, 95%
to 98% for accuracy, 92% to 100% for structure and 84% to 98% for adaptability,
the lowest on happiness under full answers, where the adaptability of
Gemma-4-31B-it is zero in 23% of the replicates. The percentile interval alone
holds it in 0% to 40% for accuracy and 0% to 81% for adaptability, and the
percentiles less the shift alone in 72% to 97% for PFS. The interval holds the
`population` difference far less often for accuracy (0% to 23%): a score computed
on small cells is itself shifted from the score of the tables the cells were
drawn from, and the interval describes the score as computed without correcting
it.

In [9]:
shown = ["observed", "shift", "low", "high", "percentile_low", "percentile_high", "share_tied", "separated"]
pfs = differences[differences["reading"].eq("pfs")].set_index(["study", "question", "mode", "model", "reference"])
print(pfs[shown].round(3).to_string())
covered = ["study", "question", "mode", "model", "reading", "expected", "shift_mean", "interval_covers_expected", "percentile_covers_expected", "centered_covers_expected", "interval_excludes_zero"]
print(calibrated[covered].round(3).to_string(index=False))

                                                                 observed  shift    low   high  percentile_low  percentile_high  share_tied  separated
study     question     mode model                reference                                                                                            
synthetic synthetic    fa   B                    A                  0.024  0.007 -0.025  0.078          -0.018            0.078       0.000      False
                            C                    A                  0.095 -0.008  0.045  0.136           0.045            0.128       0.000       True
wvs       d_happy      fa   Qwen3-VL-8B-Thinking Gemma-4-31B-it     0.341  0.044  0.275  0.422           0.319            0.422       0.000       True
                       ntp  Qwen3-VL-8B-Thinking Gemma-4-31B-it     0.273  0.030  0.209  0.386           0.239            0.386       0.000       True
          d_polpos     fa   Qwen3-VL-8B-Thinking Gemma-4-31B-it     0.080  0.069 -0.152  0.339